## Test Data Parser

In [ ]:
import os
import pandas as pd
pd.set_option('display.max_columns', None)
import pickle

from oligo_designer_toolsuite.utils import GffParser, parse_fasta_header

%load_ext memory_profiler

In [ ]:
# Initialize parser
parser = GffParser()

In [ ]:
##### Test GFF3 parser
file_gff = os.path.join(os.path.dirname(os.getcwd()), "data/custom_GCF_000001405.40_GRCh38.p14_genomic_chr16.gff")
dataframe_gff = parser.read_gff(file_gff, target_lines=10)

assert dataframe_gff.shape[1] == 23, "error: GFF3 dataframe not correctly loaded"

In [ ]:
file_gff = "/Users/lisa.barros/projects/GP0002_Oligo_Designer_Toolsuite/Toolsuite/oligo-designer-toolsuite/tests/output/GCF_000001405.40_GRCh38.p14_genomic.gtf"
target_lines = 100000

In [ ]:
%mprun -f parser.read_gff parser.read_gff(file_gff, target_lines=target_lines)

In [ ]:
# Initialize parser
parser = GffParser()

In [ ]:
##### Test GTF parser
file_gtf = os.path.join(os.path.dirname(os.getcwd()), "data/custom_GCF_000001405.40_GRCh38.p14_genomic_chr16.gtf")
dataframe_gtf = parser.read_gff(file_gtf, target_lines=10)

assert dataframe_gtf.shape[1] == 20, "error: GTF dataframe not correctly loaded"

## Test Fasta Header Parser

In [ ]:
header = "ARPG3::transcript_id=XM4581;exon_id=XM4581_exon1::16:70265537-70265662(-)"
region, additional_information, coordinates = parse_fasta_header(header)
assert region == "ARPG3", "error: wrong region parsed"
assert coordinates["chromosome"] == ["16"], "error: wrong chrom parsed"
assert coordinates["start"] == [70265537], "error: wrong start parsed"
assert coordinates["end"] == [70265662], "error: wrong end parsed"
assert coordinates["strand"] == ["-"], "error: wrong strand parsed"
assert (
    additional_information == "transcript_id=XM4581;exon_id=XM4581_exon1"
), "error: wrong additional information parsed"

## Test VCF parser

In [1]:
import os
import subprocess
from cyvcf2 import VCF

In [ ]:
file_vcf = "./SNP_testfiles/GCF_000001405.40.gz"
file_vcf_chr16_tmp = "./SNP_testfiles/GCF_000001405.40.chr16_tmp.vcf"
file_vcf_chr16 = "./SNP_testfiles/GCF_000001405.40.chr16.vcf"
file_chr_mapping = "./SNP_testfiles/chr_mapping.txt"

chr16_refseq_accession = "NC_000016.10"

In [ ]:
cmd = f"bcftools view -r {chr16_refseq_accession} --output-file {file_vcf_chr16_tmp} --output-type v {file_vcf}"
process = subprocess.Popen(cmd, shell=True, cwd="./", stdout=subprocess.DEVNULL).wait()

[W::vcf_parse_info] Extreme INFO/RS value encountered and set to missing at NC_000016.10:566081


In [ ]:
cmd = f"bcftools annotate --rename-chrs {file_chr_mapping} -o {file_vcf_chr16} -O v {file_vcf_chr16_tmp}"
process = subprocess.Popen(cmd, shell=True, cwd="./", stdout=subprocess.DEVNULL).wait()

In [5]:
vcf = VCF(file_vcf_chr16)
variant_type_identifier = "SNV"

In [9]:
for i, variant in enumerate(vcf): 

    variant_type = variant.INFO.get('VC')

    if variant_type == variant_type_identifier:

        #print(str(variant))
        print(f"REF:{variant.REF}")
        print(f"ALT:{variant.ALT}")

        print(f"CHROM:{variant.CHROM}")
        print(f"start:{variant.start}")
        print(f"end:{variant.end}")

        print(f"ID:{variant.ID}")
        print(f"VC:{variant_type}")

        print("\n")

    if i > 5:
        break

REF:T
ALT:['G']
CHROM:16
start:10166
end:10167
ID:rs1896972751
VC:SNV


REF:T
ALT:['C']
CHROM:16
start:10167
end:10168
ID:rs1207290905
VC:SNV


REF:G
ALT:['A']
CHROM:16
start:10168
end:10169
ID:rs558005370
VC:SNV


REF:G
ALT:['A', 'T']
CHROM:16
start:10169
end:10170
ID:rs1263140383
VC:SNV


REF:G
ALT:['A', 'C', 'T']
CHROM:16
start:10170
end:10171
ID:rs1037473004
VC:SNV


REF:A
ALT:['C']
CHROM:16
start:10171
end:10172
ID:rs1282658436
VC:SNV




In [67]:
import pandas as pd
import math
import numpy as np
import itertools
file_readout_probe_table = "../../data/predefined_sequences/readout_probes.tsv"

In [107]:
def load_readout_probe_table(file_readout_probe_table: str):

    required_cols = ["bit", "channel", "readout_probe_id", "readout_probe_sequence", "L/R"]

    readout_probe_table = pd.read_csv(file_readout_probe_table, sep=None, engine="python")

    # Check if all required columns exist in readout_probe_table
    cols = set(readout_probe_table.columns)
    if set(required_cols).issubset(cols):
        print("All required columns are present!")
    else:
        missing = set(required_cols) - cols
        raise ValueError(f"Missing columns: {missing}")
    
    readout_probe_table = readout_probe_table.sort_values(by=["readout_probe_id", "channel"])
    readout_probe_table.reset_index(inplace=True, drop=True)
    readout_probe_table["bit"] = "bit_" + (readout_probe_table.index + 1).astype(str)

    return readout_probe_table[required_cols]

readout_probe_table = load_readout_probe_table(file_readout_probe_table)

All required columns are present!


In [108]:
readout_probe_table

,bit,channel,readout_probe_id,readout_probe_sequence,L/R
0,bit_1,488,1,TCATCTCAGTTAGT,L
1,bit_2,488,1,TACTTACTCAGCAC,R
2,bit_3,546,1,TCTAACATCGTGGA,L
3,bit_4,546,1,TACCAGAATAACGG,R
4,bit_5,647,1,AATAGCGGAACTTG,L
...,...,...,...,...,...
175,bit_176,488,30,CGGAATGGATTGTT,R
176,bit_177,546,30,CCACTCAGAGATTA,L
177,bit_178,546,30,CAACCACCTAATGA,R
178,bit_179,647,30,TGACACATTCTACG,L


In [109]:
readout_probe_table["channel"].unique()

array([488, 546, 647])

In [117]:
n_channels = readout_probe_table["channel"].unique()
n_readout_probes_R = readout_probe_table["L/R"].value_counts()["R"]
n_readout_probes_L = readout_probe_table["L/R"].value_counts()["L"]
n_readout_probes_LR = int(min([n_readout_probes_R, n_readout_probes_L]))
n_readout_probes_LR

90

In [ ]:
n_regions = 100
n_channels = 3
n_readout_probes_LR = 30


def _generate_barcode(combination: set, codebook_size: int) -> list:
    index1 = ((n_channels * 2) * combination[0]) + (2 * combination[2])
    index2 = ((n_channels * 2) * combination[1]) + (2 * combination[2]) + 1
    barcode = np.zeros(codebook_size, dtype=np.int8)
    barcode[[index1, index2]] = 1
    return barcode

codebook = []
codebook_size = n_channels * n_readout_probes_LR * 2

combinations = list(itertools.product(list(range(n_readout_probes_LR)), list(range(n_readout_probes_LR)), list(range(n_channels))))
combinations = sorted(combinations, key=lambda t: (0 if t[0] == t[1] else 1, t[1]))
codebook_size_max = len(combinations)


if codebook_size_max < (2 * n_regions):
    raise ValueError(
        f"The number of valid barcodes ({codebook_size_max}) is lower than the required number of readout probes ({2 * n_regions}) for {n_regions} regions. Consider increasing the number of L/R readout probes."
    )

for combination in combinations[:n_regions]:
    barcode = _generate_barcode(
        combination=combination,
        codebook_size=codebook_size,
    )
    codebook.append(barcode)

codebook = pd.DataFrame(codebook, columns=[f"bit_{i+1}" for i in range(codebook_size)])
codebook

,bit_1,bit_2,bit_3,bit_4,bit_5,bit_6,bit_7,bit_8,bit_9,bit_10,...,bit_171,bit_172,bit_173,bit_174,bit_175,bit_176,bit_177,bit_178,bit_179,bit_180
0,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
98,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
def get_indices(a, b):
    """
    Computes (i1, i2) from (a,b) according to:
      a=0,b=0 -> (0,1)
      a=0,b=1 -> (3,4)
      a=0,b=2 -> (5,6)
      a=1,b=0 -> (7,8)
      a=1,b=1 -> (9,10)
      a=1,b=2 -> (11,12)
      a=2,b=0 -> (13,14)

    Uses the formula:
      index = 3*a + b
      i1 = 0 if index==0 else 2*index + 1
      i2 = i1 + 1

    Returns (i1, i2).
    """
    i1 = 3 * a + 2 * b
    
    i2 = i1 + 1
    return i1, i2

print(get_indices(0, 0))
print(get_indices(0, 1))
print(get_indices(0, 2))
print(get_indices(1, 0))
print(get_indices(1, 1))
print(get_indices(1, 2))

(0, 1)
(2, 3)
(4, 5)
(3, 4)
(5, 6)
(7, 8)
